# 04 · Llamadas A/V con WebRTC

**Objetivo**: entender cómo dos peers que solo comparten un canal TCP de mensajería terminan intercambiando audio cifrado **directo UDP↔UDP** sobre internet.

Cubre la capa 5. Fuente: [src/call.ts](../../src/call.ts). Librería: [werift](https://github.com/shinyoshiaki/werift-webrtc) (WebRTC puro en TypeScript).

## 1. Por qué TCP no sirve para audio

TCP garantiza orden y entrega. Para conseguirlo, **retransmite paquetes perdidos** y **bloquea los siguientes hasta que llegue el faltante** (head-of-line blocking).

Consecuencia en una llamada:

```
Frame 1 (20ms audio) → ✓ entrega
Frame 2 (20ms audio) → 💥 paquete perdido
Frame 3 (20ms audio) → llegó pero ESPERA
Frame 4 (20ms audio) → idem
(...500ms después, retransmisión llega)
Frame 2 → ✓
Frame 3 → ✓ (al instante)
Frame 4 → ✓
```

El usuario oye **medio segundo de silencio** seguido de un *brrrrrr* acelerado al recuperar el atasco. Inutilizable.

**La regla de oro en tiempo real**: mejor escuchar un click (pérdida) que escuchar todo retrasado. Por eso WebRTC usa **UDP**.

Pero UDP no resuelve por sí solo dos problemas:

1. **¿Cómo atravesar NAT?** El UDP sale, pero ¿cómo le hablo a un peer detrás de NAT?
2. **¿Cómo cifrar?** UDP no es TLS por defecto.

WebRTC = UDP + ICE (NAT traversal) + DTLS (handshake de claves) + SRTP (media cifrada).

## 2. Las tres capas de WebRTC

```
Audio Opus encapsulado en RTP
        ▼
  ┌─────────────────────────┐
  │  SRTP (RFC 3711)        │  ← cifra cada paquete RTP con AES-CM
  ├─────────────────────────┤
  │  DTLS (RFC 6347)        │  ← acuerda claves SRTP al inicio
  ├─────────────────────────┤
  │  ICE (RFC 8445)         │  ← decide qué camino UDP usar
  ├─────────────────────────┤
  │  UDP                    │
  └─────────────────────────┘
```

werift implementa todo esto en TypeScript. Nosotros no escribimos nada de bajo nivel.

### ICE en detalle

Cada peer recolecta **candidatos** (IP:puerto por los que se le puede contactar):

| Tipo | De dónde sale | Cuándo funciona |
|------|---------------|------------------|
| **host** | IP local de cada interfaz | Misma LAN |
| **srflx** | IP pública vista por un STUN | NAT sencillo (cone NAT) |
| **prflx** | Aprendido durante checks ICE | Edge cases |
| **relay** | IP de un TURN que retransmite | NAT simétrico (último recurso) |

Cada candidato se manda al peer remoto **vía señalización** (en nuestro caso `CALL_ICE`). Una vez tienen los candidatos del otro, ICE **prueba pares** con mensajes STUN binding hasta encontrar uno que pase. Ese par gana y se usa para el resto de la sesión.

**Nuestro setup**: solo STUN (descubrimiento de srflx), no TURN. Limitación pedagógica conocida — si ambos peers están detrás de NAT simétrico, no hay forma de conectar. Para uso real necesitarías auto-hospedar [coturn](https://github.com/coturn/coturn).

### DTLS

Una vez ICE encuentra el par, se hace un handshake DTLS:

- Cada peer genera un certificado **autofirmado al vuelo** (no requiere PKI).
- Las huellas SHA-256 de esos certificados se publican en la SDP.
- Como la SDP llega por la señalización **fiable** (nuestro TCP P2P), un MITM en UDP sería detectado por mismatch de huella.

El handshake DTLS deriva un par de claves SRTP (master key + master salt).

### SRTP

Cada paquete RTP saliente se cifra con AES-CM 128 + tag HMAC-SHA1 80-bit. El receptor descifra con la misma clave. werift hace esto transparentemente.

## 3. Señalización vs media: la separación clave

WebRTC tiene una idea fundamental: **señalización y media son canales distintos**.

- **Señalización**: SDP (descripción de sesión) + candidatos ICE. Texto, pocos bytes, solo durante el setup. *Cualquier* canal sirve: HTTP, WebSocket, paloma mensajera.
- **Media**: paquetes RTP cifrados con SRTP. UDP, alta tasa, durante toda la llamada.

WebRTC NO especifica cómo se hace la señalización — es decisión de la aplicación.

En p2p-chat usamos **nuestro propio TCP P2P** como señalización (mensajes `CALL_OFFER/ANSWER/ICE/END`). Conveniente: ya estaba ahí.

Diagrama del flujo completo:

```
caller A                                callee B
────────                                ────────
createOffer()
setLocalDescription(offer)         ── ICE gathering arranca
                                                            
                CALL_OFFER ────────►   setRemoteDescription(offer)
                                       createAnswer()
                                       setLocalDescription(answer)
                ◄──── CALL_ANSWER     ── ICE gathering arranca
setRemoteDescription(answer)
                                                            
  ◄──── CALL_ICE ────────►  CALL_ICE ────►   (trickle bidireccional)
                                                            
  [ICE: prueba pares de candidatos hasta encontrar uno que pase]
                                                            
  [DTLS handshake sobre el par UDP elegido → deriva claves SRTP]
                                                            
  connectionState=connected               connectionState=connected
                                                            
  ═══════════ Audio RTP cifrado (SRTP) UDP↔UDP ═══════════
                                                            
                                                            
CALL_END ────────►                       (cleanup)
```

Lo importante: **una vez `connected`, el TCP de señalización solo se usa para `CALL_END`**. Todo el audio va por UDP directo.

## 4. Anatomía de un paquete RTP

RTP (RFC 3550) es el formato que viaja dentro de SRTP:

```
 0                   1                   2                   3
 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|V=2|P|X|  CC   |M|     PT      |       sequence number         |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|                           timestamp                           |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|           synchronization source (SSRC) identifier            |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|                       payload (Opus)                          |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
```

Vamos a parsear uno manualmente:

In [ ]:
import struct

def parse_rtp(buf):
    b0, b1, seq, ts, ssrc = struct.unpack('>BBHII', buf[:12])
    version = (b0 >> 6) & 0b11
    padding = (b0 >> 5) & 0b1
    extension = (b0 >> 4) & 0b1
    csrc_count = b0 & 0b1111
    marker = (b1 >> 7) & 0b1
    pt = b1 & 0b1111111
    return {
        'version': version,
        'padding': bool(padding),
        'extension': bool(extension),
        'csrc_count': csrc_count,
        'marker': bool(marker),
        'payload_type': pt,
        'sequence': seq,
        'timestamp': ts,
        'ssrc_hex': f'0x{ssrc:08x}',
        'payload_len': len(buf) - 12,
    }

# Ejemplo: paquete Opus realista
# V=2, P=0, X=0, CC=0 → 0x80
# M=0, PT=96 → 0x60
# seq=1234, ts=480000, ssrc=0xCAFEBABE
rtp = bytes([0x80, 0x60]) + struct.pack('>HII', 1234, 480000, 0xCAFEBABE) + b'\x00' * 80
print('hex:', rtp.hex())
print('parse:', parse_rtp(rtp))

### Variante con librería: `aiortc` (WebRTC en Python — abstrae todo el stack RTP)

```
pip install aiortc
```

El parser de RTP que acabamos de escribir existe en `aiortc` —y con él ICE, DTLS, SRTP y el jitter buffer— todo listo. Lo que antes requería parsear headers manualmente, ahora se reduce a `addTrack()` + `createOffer()` + intercambiar SDP.

```python
from aiortc import RTCPeerConnection, RTCSessionDescription
from aiortc.contrib.media import MediaPlayer, MediaRecorder
import asyncio

# ─── Caller (peer A) ─────────────────────────────────────────────────────────
async def caller(signaling_send, signaling_recv):
    pc = RTCPeerConnection()

    # Fuente de audio: archivo WAV/OGG para tests, o micrófono en producción
    player = MediaPlayer("audio.wav")           # o MediaPlayer("default", format="pulse")
    pc.addTrack(player.audio)                   # Opus encode + RTP + SRTP + ICE

    @pc.on("icecandidate")
    async def on_ice(candidate):
        if candidate:
            await signaling_send({"type": "ice", "candidate": str(candidate)})

    offer = await pc.createOffer()
    await pc.setLocalDescription(offer)
    await signaling_send({"type": "offer", "sdp": pc.localDescription.sdp})

    # Recibir answer + candidatos ICE via señalización (nuestro TCP P2P)
    answer_msg = await signaling_recv()
    await pc.setRemoteDescription(RTCSessionDescription(answer_msg["sdp"], "answer"))

    # ICE conecta → DTLS handshake → SRTP media fluye
    await asyncio.sleep(30)   # duración de la llamada
    await pc.close()


# ─── Callee (peer B) ─────────────────────────────────────────────────────────
async def callee(signaling_send, signaling_recv):
    pc = RTCPeerConnection()
    recorder = MediaRecorder("received.wav")    # graba lo que llega

    @pc.on("track")
    async def on_track(track):
        recorder.addTrack(track)               # conecta RTP entrante al grabador
        await recorder.start()

    @pc.on("icecandidate")
    async def on_ice(candidate):
        if candidate:
            await signaling_send({"type": "ice", "candidate": str(candidate)})

    offer_msg = await signaling_recv()
    await pc.setRemoteDescription(RTCSessionDescription(offer_msg["sdp"], "offer"))
    answer = await pc.createAnswer()
    await pc.setLocalDescription(answer)
    await signaling_send({"type": "answer", "sdp": pc.localDescription.sdp})

    await asyncio.sleep(30)
    await recorder.stop()
    await pc.close()
```

**Qué hace `aiortc` por ti** (vs parsear RTP a mano):

| Capa | Manual (nuestro NB04) | `aiortc` |
|---|---|---|
| RTP header | `struct.unpack('>BBHII', buf[:12])` | Invisible — `track.recv()` devuelve frames |
| Timestamp handling | Manual, en muestras (960/frame) | Jitter buffer + resampler incluidos |
| SRTP | No implementado | AES-CM 128 automático |
| ICE | No implementado | STUN + gathering + connectivity checks |
| DTLS | No implementado | Certificado autofirmado al vuelo |
| Señalización SDP | No implementado | `createOffer/Answer` genera SDP válida |

**Para tests sin micrófono**, usa un archivo como fuente:

```python
player = MediaPlayer("audio.ogg")       # cualquier formato que soporte ffmpeg
# o para generar tono sintético:
player = MediaPlayer("sine=frequency=440", format="lavfi")
```

La señalización (`signaling_send/recv`) puede ser cualquier canal: el TCP P2P de este proyecto, un WebSocket, o incluso copiar-pegar el SDP entre terminales para un test manual.

### El timestamp avanza en *muestras*, no en ms

Para audio a 48 kHz, una trama Opus típica de 20 ms contiene **960 muestras** (48000 × 0.02). El campo `timestamp` del próximo paquete será el actual + 960.

Si el receptor recibe dos paquetes consecutivos con timestamps `t0` y `t0 + 960`, sabe que representan 20 ms reales de audio. Si recibe `t0` y `t0 + 1920`, ha perdido la trama intermedia y puede compensar (silencio, PLC — Packet Loss Concealment, etc.).

**Lección**: RTP separa el reloj de transmisión del reloj de la fuente. Esto es lo que permite mantener sincronía A/V incluso con jitter de red.

## 5. El bridge ffmpeg ↔ werift

werift sabe **mandar y recibir paquetes RTP**, pero no sabe capturar de un micrófono ni reproducir por altavoces. Esos son problemas del sistema operativo.

Solución: **ffmpeg como subproceso**.

```
  Captura                                          Reproducción
  ───────                                          ─────────────
  mic                                              altavoces
   ▼                                                ▲
  ffmpeg                                           ffplay
   ▼ encode Opus, escribe RTP a UDP                ▲ decode Opus desde RTP
  127.0.0.1:senderBridgePort                       127.0.0.1:playerBridgePort
   ▼                                                ▲
  dgram socket (call.ts)                           dgram socket (call.ts)
   ▼ writeRtp(buf)                                  ▲ playerBridge.send(buf)
  ┌──────────────────────────────────────────────────────────────────┐
  │ werift MediaStreamTrack                                          │
  │   ──────── SRTP/UDP (ICE elegido) ─────►   werift remoto         │
  └──────────────────────────────────────────────────────────────────┘
```

Cada bridge UDP local sirve para **traducir entre un proceso (ffmpeg) que solo habla sockets y un objeto JS (track) que solo habla `writeRtp/onReceiveRtp`**.

### Comandos ffmpeg

Sender (mic → RTP):

```bash
ffmpeg -f pulse -i default \
       -ac 2 -ar 48000 -c:a libopus -b:a 64k \
       -application voip -payload_type 96 \
       -f rtp rtp://127.0.0.1:<bridgePort>
```

Receiver (ffplay lee SDP que generamos al vuelo):

```bash
ffplay -protocol_whitelist file,rtp,udp \
       -fflags nobuffer -flags low_delay \
       -nodisp -autoexit \
       -i /tmp/p2p-chat-<callId>.sdp
```

SDP receptor:

```
v=0
o=- 0 0 IN IP4 127.0.0.1
s=p2p-chat
c=IN IP4 127.0.0.1
t=0 0
m=audio <playerBridgePort> RTP/AVP 96
a=rtpmap:96 opus/48000/2
```

## 6. Trickle ICE

ICE gathering tarda **segundos** (consultas STUN, escaneo de interfaces). Esperar a tener TODOS los candidatos antes de iniciar la conversación retrasa el setup.

**Trickle ICE** (RFC 8838): manda cada candidato al peer remoto **en cuanto sale**. El otro extremo los va aplicando con `addIceCandidate()` y los conectividad-checks empiezan inmediatamente sobre los primeros candidatos disponibles.

```
A → B: CALL_ICE { candidate: 'host 192.168.1.5' }
A → B: CALL_ICE { candidate: 'srflx 81.43.x.x' }
A → B: CALL_ICE { candidate: null }   // fin (RFC 8838 end-of-candidates)
```

Con trickle el setup de una llamada se reduce de ~5s a ~500ms en condiciones normales.

## 7. Estados de una llamada

```
              CALL_OFFER
   idle ──────────────────► signaling
                                │
                                │ CALL_ANSWER + ICE en progreso
                                ▼
                            connecting
                                │
                                │ ICE+DTLS completos
                                ▼
                              active
                                │
                                │ CALL_END / fallo de conexión
                                ▼
                              ended
```

En [src/call.ts](../../src/call.ts) la transición la dispara `connectionStateChange.subscribe(...)`. werift emite los estados nativos de WebRTC; los traducimos a nuestros 5 estados para la UI.

## 8. Limitaciones conocidas

1. **NAT simétrico**: sin TURN, peers tras NATs simétricos no conectan. Síntoma: ICE se queda en `checking` indefinidamente y luego `failed`.
2. **Sin renegociación**: cambiar codec a mitad de llamada implicaría nueva offer/answer. No implementado.
3. **Sin vídeo**: la misma máquina sirve; basta otro transceiver `kind: 'video'` y otro ffmpeg con H.264. Lo dejamos fuera del scope.
4. **STUN público**: usamos Google. Si te importa la privacidad, auto-hospédate coturn.
5. **Sin DTX explícito**: ffmpeg con `-application voip` lo activa automáticamente; con otras fuentes podrías querer `-vbr off -dtx 0` para latencia más constante.

## 9. Ejercicio (siguiente)

Ahora que entiendes la pipeline completa, el ejercicio de [EXERCISES.md](../EXERCISES.md) es: **transferencia de archivos P2P**. Hay un bonus avanzado que reutiliza la capa 5 — un `RTCDataChannel` sobre el mismo PC para mandar chunks fiables sin necesidad de TCP. Eso te da SCTP sobre DTLS sobre UDP atravesando NAT, que es el santo grial.